# Inspecting light-curve residuals from a SALT training

A trained model can look fine overall and still miss specific supernovae, bands or phases. Residuals are how we find those misses. This notebook loads a training sample and a set of fitted parameters, then works through three examples:

1. Compare one supernova to the model, band by band
2. Rank every light curve to find the worst fits
3. Stack all the residuals to look for trends with phase and wavelength

**What we plot.** For each photometric point we compute a *pull*:

$$\text{pull} = \frac{f_\text{data} - f_\text{model}}{\sigma}, \qquad \sigma^2 = \sigma_\text{data}^2 + \sigma_\text{model}^2 + (s_c\, f_\text{model})^2$$

Here $\sigma_\text{model}$ comes from the model's error surfaces and $s_c$ is the color scatter. A positive pull means the data is brighter than the model. If the model and its uncertainties are both right, pulls should look like a unit Gaussian, with mean 0 and width 1.

Run this with the `salt` environment as the kernel, and keep the notebook in the directory your config file's paths are relative to.

## Setup

Fill in these two paths before running. `conf` is the training config, and `parfile` is the fitted parameters from a training output directory (usually `salt3_parameters_unscaled.dat`). They need to go together: same sample, same `maxsn`, same model settings.

In [ ]:
conf = "BLANK"     # training config file
parfile = "BLANK2" # fitted parameters from a training run
args = ""          # extra command-line options, e.g. "--maxsn 100"

In [ ]:
import numpy as np
from astropy.table import Table
from jax import numpy as jnp
from matplotlib import pyplot as plt
from scipy import stats
from tqdm.auto import tqdm

from saltshaker.training.TrainSALT import TrainSALT, RunTraining
from saltshaker.util import readutils

%matplotlib inline

## Load the data and the model

This reads the data, applies the same cuts as training, and builds `saltresids`, the object that holds the model and the data. The first run can take several minutes.

In [ ]:
salt = TrainSALT()
RunTraining().get_config_options(salt, conf, args.split())
salt.options.resume_from_outputdir = False

salt.kcordict = readutils.rdkcor(salt.surveylist, salt.options)
datadict = readutils.rdAllData(
    salt.options.snlists,
    salt.options.estimate_tpk,
    dospec=salt.options.dospec,
    peakmjdlist=salt.options.tmaxlist,
    binspecres=salt.options.binspecres,
    snparlist=salt.options.snparlist,
    maxsn=salt.options.maxsn,
)
datadict = salt.mkcuts(datadict)[0]
x_init, saltresids = salt.initializesaltmodelobject(datadict)

## Load the fitted parameters

`x_init` is only the starting point of a fit, so we replace it with the saved values, matching by parameter name. Parameters with a repeated name, such as the `m0` spline coefficients, keep their order in the file. Finally we apply the model's constraints, which is what the training does before computing a likelihood.

In [ ]:
names, values = np.loadtxt(parfile, unpack=True, skiprows=1, dtype="U30,f8")

xfit = np.array(x_init)
for name in np.unique(saltresids.parlist):
    xfit[saltresids.parlist == name] = values[names == name]

x = saltresids.constraints.transformtoconstrainedparams(jnp.array(xfit))

## Computing pulls for one light curve

`saltresids.datadict` holds one object per supernova, and each `sn.photdata` is a dictionary of light curves keyed by band.

Two details to watch for:

- **Padding.** Light curves are padded with zeros so that JAX can batch them, and `lc.ipad` marks the fake points. Always remove them.
- **`lc.modelresidual` is not a pull.** It returns *whitened* residuals. With color scatter on, those are mixtures of several points in a light curve, so they can't be read point by point. That's why we compute the pull directly here. This also means our pulls ignore the correlation that color scatter adds between points, so treat them as a diagnostic, not the exact likelihood.

In [ ]:
def lcresiduals(lc, x):
    """Return the phase, data, model, uncertainty and pull of a light curve's real points."""
    use = ~np.asarray(lc.ipad)
    phase = np.asarray(lc.phase)[use]
    flux = np.asarray(lc.fluxcal)[use]
    fluxerr = np.asarray(lc.fluxcalerr)[use]
    model = np.asarray(lc.modelflux(x))[use]
    modelvariance = np.asarray(lc.modelfluxvariance(x))[use]
    colorscatter = float(lc.colorscatter(x))

    sigma = np.sqrt(fluxerr**2 + modelvariance + (colorscatter * model) ** 2)
    return {
        "phase": phase,
        "flux": flux,
        "fluxerr": fluxerr,
        "model": model,
        "sigma": sigma,
        "pull": (flux - model) / sigma,
    }

## Example 1: one supernova

The left column shows the data and model, with the shaded band showing the total uncertainty. The right column shows the pulls. Look for runs of same-sign pulls: a single outlier could be a bad data point, but a run usually means the model has the wrong shape there.

Change `snid` to look at another supernova.

In [ ]:
snid = list(saltresids.datadict)[0]

sn = saltresids.datadict[snid]
nbands = len(sn.photdata)
fig, axes = plt.subplots(nbands, 2, figsize=(11, 2.4 * nbands), sharex=True, squeeze=False)

for (axflux, axpull), (band, lc) in zip(axes, sn.photdata.items()):
    r = lcresiduals(lc, x)
    order = np.argsort(r["phase"])
    phase, model, sigma = r["phase"][order], r["model"][order], r["sigma"][order]

    axflux.errorbar(r["phase"], r["flux"], yerr=r["fluxerr"], fmt="k.", label="data")
    axflux.plot(phase, model, "r-", label="model")
    axflux.fill_between(phase, model - sigma, model + sigma, color="r", alpha=0.2)
    axflux.set_ylabel("flux")
    axflux.set_title(f"{band}, chi2/point = {(r['pull']**2).mean():.2f}", fontsize=10)

    axpull.axhline(0, color="k", lw=1)
    axpull.axhspan(-1, 1, color="0.9")
    axpull.plot(r["phase"], r["pull"], "k.")
    axpull.set_ylabel("pull")

axes[0, 0].legend()
axes[-1, 0].set_xlabel("phase (days)")
axes[-1, 1].set_xlabel("phase (days)")
fig.suptitle(f"SN {snid}, z = {sn.zHelio:.3f}")
fig.tight_layout()

## Example 2: finding the worst light curves

Now we do the same for every light curve and collect the results. This calls the model once per light curve, so it can take a few minutes on a large sample. Everything later uses `allpulls`, so you only need to run this once.

In [ ]:
summary, allpulls = [], []
for snid, sn in tqdm(saltresids.datadict.items()):
    for band, lc in sn.photdata.items():
        r = lcresiduals(lc, x)
        if not len(r["pull"]):
            continue
        summary.append({
            "sn": snid,
            "survey": sn.survey,
            "band": band,
            "npoints": len(r["pull"]),
            "chi2perpoint": (r["pull"] ** 2).mean(),
            "meanpull": r["pull"].mean(),
        })
        allpulls.append({
            "phase": r["phase"],
            "restwave": np.full(len(r["pull"]), float(lc.lambdaeffrest)),
            "pull": r["pull"],
        })

summary = Table(rows=summary)
allpulls = {key: np.concatenate([pulls[key] for pulls in allpulls]) for key in allpulls[0]}

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(summary["chi2perpoint"], bins=np.logspace(-1.5, 2, 40), color="0.5")
ax.axvline(1, color="r", ls="--", label="expected")
ax.set_xscale("log")
ax.set_xlabel("chi2/point of each light curve")
ax.set_ylabel("number of light curves")
ax.legend()

# ignore very short light curves, where chi2/point is noisy
longenough = summary[summary["npoints"] >= 5]
longenough[np.argsort(-longenough["chi2perpoint"])][:10]

Take an SN from this table, put it in `snid` in Example 1, and see what went wrong. It's also worth grouping by survey: if one survey stands out, the cause is more likely calibration or its uncertainties than the model.

In [ ]:
bysurvey = Table(
    rows=[
        {
            "survey": survey,
            "lightcurves": (summary["survey"] == survey).sum(),
            "medianchi2": np.median(summary["chi2perpoint"][summary["survey"] == survey]),
            "meanpull": np.mean(summary["meanpull"][summary["survey"] == survey]),
        }
        for survey in np.unique(summary["survey"])
    ]
)
bysurvey[np.argsort(-bysurvey["medianchi2"])]

## Example 3: stacking all the residuals

A single light curve can be noisy, but a small trend shared by many supernovae shows up clearly when they're stacked. This is how to spot model problems instead of problems with individual supernovae. The red points show the mean pull in bins of phase and of rest-frame wavelength, and they should sit near zero.

In [ ]:
def binnedmean(values, pulls, nbins=15):
    """Mean pull and its standard error in equal-count bins of values."""
    edges = np.unique(np.quantile(values, np.linspace(0, 1, nbins + 1)))
    indices = np.clip(np.digitize(values, edges) - 1, 0, edges.size - 2)
    binned = [indices == i for i in np.unique(indices)]
    centers = [values[i].mean() for i in binned]
    means = [pulls[i].mean() for i in binned]
    errors = [pulls[i].std() / np.sqrt(i.sum()) for i in binned]
    return centers, means, errors


fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)
for ax, column, label in zip(axes, ["phase", "restwave"], ["phase (days)", "rest-frame wavelength (Angstroms)"]):
    ax.plot(allpulls[column], allpulls["pull"], "k.", ms=1, alpha=0.15, rasterized=True)
    centers, means, errors = binnedmean(allpulls[column], allpulls["pull"])
    ax.errorbar(centers, means, yerr=errors, fmt="ro")
    ax.axhline(0, color="k", lw=1)
    ax.set_xlabel(label)
axes[0].set_ylim(-5, 5)
axes[0].set_ylabel("pull")
fig.tight_layout()

Finally, the width of the pulls tells us about the uncertainties. A distribution wider than the unit Gaussian means the errors are underestimated, or the model is missing something. A narrower one means the errors are overestimated. The tails show the outliers that the width of a Gaussian can't describe.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
bins = np.linspace(-6, 6, 61)
ax.hist(allpulls["pull"], bins=bins, density=True, color="0.6", label="pulls")
ax.plot(bins, stats.norm.pdf(bins), "r-", label="unit Gaussian")
ax.set_yscale("log")
ax.set_ylim(1e-4, 1)
ax.set_xlabel("pull")
ax.legend()

pulls = allpulls["pull"]
print(f"mean {pulls.mean():.3f}, std {pulls.std():.3f}")
print(f"robust width (from the 16th and 84th percentiles) {np.diff(np.percentile(pulls, [16, 84]))[0] / 2:.3f}")
print(f"fraction beyond 5 sigma {(np.abs(pulls) > 5).mean():.4f}")

## Things to try next

- Make the Example 3 plots for a single survey or band. Does a trend appear in one place only?
- Split the sample by redshift. The same rest-frame phase and wavelength come from different filters at different redshifts, so this helps separate model problems from filter problems.
- Is the phase trend the same in blue and red bands? Try binning in phase within a range of `restwave`.
- Pick an outlier from Example 2 and work out whether it's one bad point or the whole light curve.